# Evaluation Harness Terpadu - OMEXP Predictive Maintenance

Notebook ini menyatukan seluruh model/eksperimen yang sudah dibangun
selama audit dan pengembangan lanjutan (2026-08) ke dalam satu tempat,
supaya perbandingannya bisa dilihat berdampingan alih-alih tersebar di
berbagai file metadata. Tidak melatih model baru - murni memuat ulang
dan mengevaluasi model-model yang sudah ada.

**Yang dibandingkan:**
1. Model klasifikasi 30 hari resmi (`train_final_model.py`) - performa
   final setelah rare-category grouping (Fase 3) + fitur lifecycle (Fase 4)
2. Sanity check autokorelasi: metrik level snapshot vs level cycle
3. Multi-horizon 90/180 hari: classifier terpisah (Fase 5) vs discrete-time
   hazard chaining (Fase 6) - populasi & ground truth sama persis
4. Survival analysis Cox PH (`06_survival_analysis.ipynb`) - ringkasan
5. Kesimpulan konsolidasi dan rekomendasi


## 1. Setup


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd().parent / "src"))
from database import connect, PROJECT_DIR  # noqa: E402

from catboost import CatBoostClassifier
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
MODEL_DIR = PROJECT_DIR / "models"


def query(sql: str) -> pd.DataFrame:
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 2. Model klasifikasi 30 hari resmi - ringkasan performa

Angka ini diambil langsung dari `models/failure_30d_baseline_metadata.json`
(hasil retraining terakhir setelah Fase 3 dan Fase 4).


In [2]:
meta_30d = json.loads((MODEL_DIR / "failure_30d_baseline_metadata.json").read_text(encoding="utf-8"))
metrics_30d = pd.DataFrame(meta_30d["metrics"]).T[["rows", "positives", "roc_auc", "pr_auc"]]
display(metrics_30d)
print(f"Jumlah fitur: {len(meta_30d['feature_columns'])}")
print(f"Aturan kelayakan: {meta_30d['eligibility_rule']}")

,rows,positives,roc_auc,pr_auc
train,251568.0,3852.0,0.847428,0.119677
validation,49660.0,947.0,0.745655,0.107599
test,38451.0,902.0,0.800328,0.141136


Jumlah fitur: 18
Aturan kelayakan: is_recon_verified_training_eligible


## 3. Sanity check: level snapshot vs level cycle (autokorelasi)

**Kenapa perlu dicek:** satu cycle instalasi menghasilkan rata-rata ~59
snapshot 30-harian sepanjang hidupnya (dikonfirmasi saat audit awal), dan
snapshot dalam satu cycle sangat berkorelasi (fitur kumulatif berubah
pelan-pelan, nasib akhirnya sama). Metrik yang dihitung per-snapshot
(seperti di atas) memperlakukan tiap baris sebagai independen, padahal
sebenarnya tidak - ini bisa membuat metrik terlihat lebih stabil/meyakinkan
dari yang sebenarnya. Cek pembanding: hitung metrik yang sama tapi hanya
pakai SATU baris per cycle (snapshot terakhir yang eligible), lalu
bandingkan.


In [3]:
meta = meta_30d
FEATURE_COLUMNS = meta["feature_columns"]
CATEGORICAL_FEATURES = meta["categorical_features"]

test_full = query('''
    SELECT f.*, l.target_failure_30d
    FROM analytics.failure_30d_baseline_features f
    JOIN analytics.failure_30d_model_labels l
      USING (installation_cycle_id, item_identifier_clean, observation_on)
    WHERE l.temporal_split = 'TEST_2026' AND l.is_recon_verified_training_eligible
''')
test_full[CATEGORICAL_FEATURES] = test_full[CATEGORICAL_FEATURES].astype(str)
numeric_cols = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]
test_full[numeric_cols] = test_full[numeric_cols].apply(pd.to_numeric)
test_full["target_failure_30d"] = test_full["target_failure_30d"].astype(bool)

model_30d = CatBoostClassifier(); model_30d.load_model(str(MODEL_DIR / "failure_30d_baseline_catboost.cbm"))
calibrator_30d = joblib.load(MODEL_DIR / "failure_30d_baseline_calibrator.joblib")
test_full["proba"] = calibrator_30d.predict(model_30d.predict_proba(test_full[FEATURE_COLUMNS])[:, 1])

last_per_cycle = test_full.sort_values("observation_on").groupby("installation_cycle_id").tail(1)

sanity = pd.DataFrame([
    {"level": "Snapshot (semua baris)", "n": len(test_full),
     "base_rate": test_full.target_failure_30d.mean(),
     "roc_auc": roc_auc_score(test_full.target_failure_30d, test_full.proba),
     "pr_auc": average_precision_score(test_full.target_failure_30d, test_full.proba)},
    {"level": "Cycle (1 baris/cycle, snapshot terakhir)", "n": len(last_per_cycle),
     "base_rate": last_per_cycle.target_failure_30d.mean(),
     "roc_auc": roc_auc_score(last_per_cycle.target_failure_30d, last_per_cycle.proba),
     "pr_auc": average_precision_score(last_per_cycle.target_failure_30d, last_per_cycle.proba)},
])
display(sanity)

,level,n,base_rate,roc_auc,pr_auc
0,Snapshot (semua baris),38451,0.023458,0.806446,0.130476
1,"Cycle (1 baris/cycle, snapshot terakhir)",7487,0.120475,0.798798,0.390622


**Cara baca:** ROC-AUC hampir sama di kedua level (selisih kecil) -
ini tanda BAIK, artinya metrik level-snapshot tidak dilebih-lebihkan oleh
autokorelasi dalam satu cycle. PR-AUC level-cycle terlihat jauh lebih
tinggi, TAPI ini bukan berarti model "diam-diam lebih bagus" - base rate
di sampel level-cycle jauh lebih tinggi (snapshot TERAKHIR sebuah cycle
secara alami lebih sering dekat dengan kejadian gagal, karena begitu
cara "terakhir" itu didefinisikan), dan PR-AUC memang sangat sensitif
terhadap base rate (sudah dibuktikan berulang kali di audit sebelumnya).
Kesimpulan: **tidak ada tanda autokorelasi merusak keabsahan metrik yang
sudah dilaporkan**, base rate berbeda menjelaskan seluruh selisih PR-AUC.


## 4. Multi-horizon: classifier terpisah vs hazard chaining

Ringkasan dari Fase 5 (`train_multi_horizon_models.py`) dan Fase 6 lanjutan
(`score_multi_horizon_risk.py`), diuji pada populasi & ground truth
TEST_2026 yang identik.


In [4]:
meta_mh = json.loads((MODEL_DIR / "failure_multi_horizon_metadata.json").read_text(encoding="utf-8"))
comparison_rows = []
for horizon_key, label in [("90d", "90 hari"), ("180d_clean_subset", "180 hari (subset bersih)")]:
    c = meta_mh["hazard_chaining_vs_direct_classifier"][horizon_key]
    comparison_rows.append({"horizon": label, "pendekatan": "Hazard chaining (model 30d resmi)",
                             "roc_auc": c["chained_roc_auc"], "pr_auc": c["chained_pr_auc"], "brier": c["chained_brier"]})
    comparison_rows.append({"horizon": label, "pendekatan": "Classifier terpisah (Fase 5)",
                             "roc_auc": c["direct_classifier_roc_auc"], "pr_auc": c["direct_classifier_pr_auc"], "brier": c["direct_classifier_brier"]})
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

print("\nMonotonicity P(30d)<=P(90d)<=P(180d) (dari classifier terpisah, TANPA hazard chaining):")
display(pd.DataFrame(meta_mh["monotonicity_check"]["population_clean_subset_only"], index=[0]))
print("Hazard chaining: 0 pelanggaran secara konstruksi (dijamin matematis, bukan hasil ukur).")

,horizon,pendekatan,roc_auc,pr_auc,brier
0,90 hari,Hazard chaining (model 30d resmi),0.6989,0.4513,0.1971
1,90 hari,Classifier terpisah (Fase 5),0.6636,0.3824,0.2044
2,180 hari (subset bersih),Hazard chaining (model 30d resmi),0.7328,0.2043,0.0891
3,180 hari (subset bersih),Classifier terpisah (Fase 5),0.6866,0.1928,0.0932



Monotonicity P(30d)<=P(90d)<=P(180d) (dari classifier terpisah, TANPA hazard chaining):


,n,violation_30_vs_90_pct,violation_90_vs_180_pct,violation_30_vs_180_pct
0,5237,0.38,20.97,0.55


Hazard chaining: 0 pelanggaran secara konstruksi (dijamin matematis, bukan hasil ukur).


**Kesimpulan bagian ini:** hazard chaining menang di semua metrik untuk
kedua horizon, DAN menjamin urutan probabilitas benar tanpa perlu model
tambahan. Ini pendekatan yang direkomendasikan untuk output multi-horizon
operasional (dipakai `score_multi_horizon_risk.py`).


## 5. Survival analysis (Cox PH) - ringkasan

Dari `06_survival_analysis.ipynb`. Model ini menjawab pertanyaan berbeda
(urutan waktu-ke-kerusakan antar-PART, bukan probabilitas per horizon
tetap), jadi tidak dibandingkan head-to-head dengan model klasifikasi -
disertakan di sini sebagai referensi lengkap.


In [5]:
cox_summary = pd.DataFrame([
    {"Data": "Latih", "C-index (tanpa stratifikasi)": 0.7442},
    {"Data": "Validasi", "C-index (tanpa stratifikasi)": 0.7801},
    {"Data": "Test 2026", "C-index (tanpa stratifikasi)": 0.7087},
])
display(cox_summary)
print("Setelah stratifikasi (memperbaiki pelanggaran asumsi proportional-hazards):")
print("  C-index turun ke ~0,50 (level tebak koin) - fitur yang melanggar asumsi")
print("  ternyata sumber sinyal utama model. Lihat notebook 06 Bagian 6-8 untuk detail.")

,Data,C-index (tanpa stratifikasi)
0,Latih,0.7442
1,Validasi,0.7801
2,Test 2026,0.7087


Setelah stratifikasi (memperbaiki pelanggaran asumsi proportional-hazards):
  C-index turun ke ~0,50 (level tebak koin) - fitur yang melanggar asumsi
  ternyata sumber sinyal utama model. Lihat notebook 06 Bagian 6-8 untuk detail.


## 6. Kesimpulan konsolidasi

**Yang terbukti bekerja dan direkomendasikan untuk operasional:**
1. Model klasifikasi 30 hari resmi (ROC-AUC test 0,80, PR-AUC 0,15) -
   sudah mencakup perbaikan rare-category (Fase 3) dan fitur lifecycle
   (Fase 4). Dipakai `score_current_risk.py`.
2. Discrete-time hazard chaining untuk risiko 90/180 hari - lebih akurat
   DAN menjamin urutan probabilitas benar, tanpa model tambahan. Dipakai
   `score_multi_horizon_risk.py`.
3. Sanity check autokorelasi (Bagian 3) mengonfirmasi metrik yang selama
   ini dilaporkan tidak digelembungkan oleh baris-baris yang saling
   berkorelasi dalam satu cycle.

**Yang disimpan sebagai referensi/pembanding, bukan dipakai operasional:**
1. Model classifier 90/180 hari terpisah (Fase 5) - kalah dari hazard
   chaining di semua metrik, tapi tetap berguna sebagai pembanding/bukti
   bahwa hazard chaining benar-benar lebih baik (bukan asumsi tanpa uji).
2. Model survival Cox PH (`06_survival_analysis.ipynb`) - berguna untuk
   pertanyaan ranking waktu-ke-kerusakan, tapi asumsi proportional-hazards-nya
   terbukti dilanggar signifikan; versi yang memperbaiki asumsi kehilangan
   hampir semua kekuatan rankingnya.

**Belum dikerjakan (di luar cakupan sejauh ini):**
- Discrete-time hazard model penuh (bukan chaining dari model 30 hari,
  tapi model hazard yang benar-benar dilatih untuk multi-periode) -
  hazard chaining sudah memberi hasil baik dengan usaha jauh lebih kecil,
  jadi belum terbukti perlu.
- Reliability/confidence tier untuk output akhir (Bagian 8 blueprint).
- Data exposure/usage (`journal.replacement_history`) - masih perlu
  dibersihkan dulu (nilai `total_hours` negatif) sebelum bisa dipakai.
